# NewsLLM RAG Pipeline — End-to-End Walkthrough

This notebook walks through every stage of the RAG pipeline in order.
Each cell maps directly to one file in the `indexing/` or `generation/` folder.

**Pipeline:**
```
INDEXING                          GENERATION
────────────────────────────      ──────────────────────────
1. Load     sample_news.json      5. Embed query
2. Chunk    split into pieces     6. Retrieve  vector search
3. Embed    text → vectors        7. Generate  LLM + context
4. Store    vectors → Qdrant      8. Response  citations
```

**Requirements — run once:**
```bash
uv add qdrant-client sentence-transformers openai python-dotenv
```

## 0. Setup

In [ ]:
import json
from datetime import datetime
from dataclasses import dataclass, field
from dotenv import load_dotenv
import os

load_dotenv()  # reads OPENAI_API_KEY from .env

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Add OPENAI_API_KEY to your .env file"
print("✓ API key loaded")

---
## INDEXING PIPELINE

### Stage 1 — Load
`indexing/loaders/file_loader.py`

Read raw documents from disk. Output: list of `RawDocument`.

In [ ]:
@dataclass
class RawDocument:
    id: str
    title: str
    text: str
    url: str
    published: datetime
    category: str = ""
    tags: list[str] = field(default_factory=list)


def load(path: str) -> list[RawDocument]:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    return [
        RawDocument(
            id=d["id"],
            title=d["title"],
            text=d["text"],
            url=d["url"],
            published=datetime.fromisoformat(d["published"]),
            category=d.get("category", ""),
            tags=d.get("tags", []),
        )
        for d in raw
    ]


docs = load("data/sample_news.json")

print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(f"  [{doc.id}] {doc.title}")

### Stage 2 — Chunk
`indexing/chunkers/semantic_chunker.py`

Split each document into smaller pieces.
**Why:** LLMs have context limits. Smaller chunks = more precise retrieval.

Key rule: every chunk must keep `title`, `url`, `published` for citation later.

In [ ]:
import re

@dataclass
class Chunk:
    chunk_id: str       # "{doc_id}_{index}"
    source_id: str
    text: str
    title: str          # preserved for citation
    url: str            # preserved for citation
    published: datetime # preserved for citation


def chunk(doc: RawDocument, chunk_size: int = 150, overlap: int = 20) -> list[Chunk]:
    """Split on Chinese sentence boundaries, cap at chunk_size characters."""
    # Split on Chinese punctuation
    sentences = re.split(r"(?<=[。！？])", doc.text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks, current, current_len = [], [], 0
    for sent in sentences:
        if current_len + len(sent) > chunk_size and current:
            chunks.append("".join(current))
            current = current[-overlap:] if overlap else []
            current_len = sum(len(s) for s in current)
        current.append(sent)
        current_len += len(sent)
    if current:
        chunks.append("".join(current))

    return [
        Chunk(
            chunk_id=f"{doc.id}_{i}",
            source_id=doc.id,
            text=text,
            title=doc.title,
            url=doc.url,
            published=doc.published,
        )
        for i, text in enumerate(chunks)
    ]


all_chunks = []
for doc in docs:
    doc_chunks = chunk(doc)
    all_chunks.extend(doc_chunks)

print(f"{len(docs)} documents → {len(all_chunks)} chunks\n")

# Inspect one document's chunks
example_chunks = [c for c in all_chunks if c.source_id == "news_001"]
print(f"news_001 split into {len(example_chunks)} chunks:")
for c in example_chunks:
    print(f"\n  [{c.chunk_id}] ({len(c.text)} chars)")
    print(f"  {c.text[:80]}...")

### Stage 3 — Embed
`indexing/embedders/bge_m3.py`

Convert each chunk's text into a vector (list of floats).
**Why:** vectors capture semantic meaning — similar meaning = similar vector.

⚠️ First run downloads the model (~500MB). Cached after that.

In [ ]:
from sentence_transformers import SentenceTransformer

# Using a lighter model for the notebook — same interface as bge-m3
# Swap to "BAAI/bge-m3" for production (better zh-tw quality, larger)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

texts = [c.text for c in all_chunks]
vectors = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"\nEmbedded {len(vectors)} chunks")
print(f"Vector dimension: {len(vectors[0])}")
print(f"First vector (first 5 values): {vectors[0][:5].tolist()}")

### Stage 4 — Store
`indexing/vector_stores/qdrant_store.py`

Save vectors + chunk metadata into Qdrant.
Using `:memory:` — no Docker needed. Data lives in RAM for this session.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# :memory: = no Docker needed. Swap to host="localhost" when using Docker.
client = QdrantClient(":memory:")

COLLECTION = "newsllm_news"
VECTOR_DIM = len(vectors[0])

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=vectors[i].tolist(),
        payload={
            "chunk_id":  chunk.chunk_id,
            "source_id": chunk.source_id,
            "text":      chunk.text,
            "title":     chunk.title,
            "url":       chunk.url,
            "published": chunk.published.isoformat(),
        },
    )
    for i, chunk in enumerate(all_chunks)
]

client.upsert(collection_name=COLLECTION, points=points)

info = client.get_collection(COLLECTION)
print(f"✓ Collection '{COLLECTION}' ready")
print(f"  {info.points_count} vectors stored, dimension={VECTOR_DIM}")

---
## GENERATION PIPELINE

### Stage 5 — Embed Query

Use the **same model** as Stage 3 to embed the user's question.
**Why:** if you use a different model here, the vectors live in different spaces
and retrieval silently breaks.

In [ ]:
query = "台積電的投資計畫是什麼？"

# Same embedder object — same model guaranteed
query_vector = embedder.encode(query, normalize_embeddings=True).tolist()

print(f"Query: {query}")
print(f"Query vector dim: {len(query_vector)}")
print(f"First 5 values: {query_vector[:5]}")

### Stage 6 — Retrieve
`generation/retrievers/hybrid_retriever.py`

Search Qdrant for the chunks most similar to the query vector.
Print scores so you can see what the model finds relevant.

In [ ]:
@dataclass
class RetrievedChunk:
    chunk: Chunk
    score: float


def retrieve(query_vector: list[float], top_k: int = 3) -> list[RetrievedChunk]:
    results = client.search(
        collection_name=COLLECTION,
        query_vector=query_vector,
        limit=top_k,
        with_payload=True,
    )
    return [
        RetrievedChunk(
            chunk=Chunk(
                chunk_id=r.payload["chunk_id"],
                source_id=r.payload["source_id"],
                text=r.payload["text"],
                title=r.payload["title"],
                url=r.payload["url"],
                published=datetime.fromisoformat(r.payload["published"]),
            ),
            score=r.score,
        )
        for r in results
    ]


retrieved = retrieve(query_vector, top_k=3)

print(f"Top {len(retrieved)} results for: '{query}'\n")
for i, r in enumerate(retrieved):
    print(f"  [{i+1}] score={r.score:.4f} | {r.chunk.title}")
    print(f"       {r.chunk.text[:100]}...\n")

### Stage 7 — Generate
`generation/llms/openai_llm.py`

Pass the retrieved chunks as context to the LLM.
Evidence-first prompt: show sources first, then ask the question.

In [ ]:
from openai import OpenAI

llm = OpenAI(api_key=OPENAI_API_KEY)


def build_prompt(query: str, chunks: list[RetrievedChunk]) -> str:
    evidence = "\n\n".join(
        f"[{i+1}] 標題：{c.chunk.title}\n"
        f"    來源：{c.chunk.url}\n"
        f"    內容：{c.chunk.text}"
        for i, c in enumerate(chunks)
    )
    return f"""你是一個可信新聞助理。請根據以下參考資料回答問題。
回答時標註來源編號 [1][2]，若資料不足請說明無法確認。

=== 參考資料 ===
{evidence}

=== 問題 ===
{query}"""


def generate(query: str, chunks: list[RetrievedChunk]) -> str:
    prompt = build_prompt(query, chunks)
    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content


answer = generate(query, retrieved)
print(answer)

### Stage 8 — Postprocess
`generation/postprocessors/citation.py`

Attach structured citations to the answer.
The `score` field is the confidence from retrieval — below a threshold we refuse to answer.

In [ ]:
@dataclass
class Citation:
    index: int
    title: str
    url: str
    published: datetime

@dataclass
class RAGResponse:
    answer: str
    citations: list[Citation]
    confidence: float
    refused: bool = False


CONFIDENCE_THRESHOLD = 0.3


def postprocess(answer: str, chunks: list[RetrievedChunk]) -> RAGResponse:
    confidence = chunks[0].score if chunks else 0.0

    if confidence < CONFIDENCE_THRESHOLD:
        return RAGResponse(
            answer="找不到足夠相關的資料來回答這個問題，建議查閱其他可信來源。",
            citations=[],
            confidence=confidence,
            refused=True,
        )

    citations = [
        Citation(
            index=i + 1,
            title=c.chunk.title,
            url=c.chunk.url,
            published=c.chunk.published,
        )
        for i, c in enumerate(chunks)
    ]
    return RAGResponse(answer=answer, citations=citations, confidence=confidence)


response = postprocess(answer, retrieved)

print("=" * 60)
print(f"ANSWER (confidence={response.confidence:.4f}, refused={response.refused})")
print("=" * 60)
print(response.answer)
print("\n--- Citations ---")
for c in response.citations:
    print(f"[{c.index}] {c.title}")
    print(f"    {c.url}")
    print(f"    {c.published.strftime('%Y-%m-%d')}")

---
## Try Your Own Query

Change the query below and re-run stages 5→8 to see how retrieval changes.

In [ ]:
# Try any of these — or write your own
# query = "健保談判的結果如何？"
# query = "台灣農業面臨什麼問題？"
# query = "捷運工程為什麼延誤？"
# query = "台灣有哪些AI新創公司？"

query = "健保談判的結果如何？"

# Run the full generation pipeline in one go
query_vector = embedder.encode(query, normalize_embeddings=True).tolist()
retrieved    = retrieve(query_vector, top_k=3)
answer       = generate(query, retrieved)
response     = postprocess(answer, retrieved)

print(f"Query: {query}\n")
print(f"Answer (confidence={response.confidence:.4f}):")
print(response.answer)
print("\nCitations:")
for c in response.citations:
    print(f"  [{c.index}] {c.title} — {c.url}")

---
## What Each Cell Maps To

| Notebook cell | Pipeline file |
|---|---|
| Stage 1 Load | `indexing/loaders/file_loader.py` |
| Stage 2 Chunk | `indexing/chunkers/semantic_chunker.py` |
| Stage 3 Embed | `indexing/embedders/bge_m3.py` |
| Stage 4 Store | `indexing/vector_stores/qdrant_store.py` |
| Stage 5 Embed query | `generation/retrievers/hybrid_retriever.py` |
| Stage 6 Retrieve | `generation/retrievers/hybrid_retriever.py` |
| Stage 7 Generate | `generation/llms/openai_llm.py` |
| Stage 8 Postprocess | `generation/postprocessors/citation.py` |

When a cell works the way you want → copy that logic into the corresponding file.
Nothing is thrown away.